# Week 4 Assignment — Building a Basic AI-Powered Search for Data Dynamics

**Course:** Outskill — Gen AI Engineering Fellowship
**Module:** RAG Sessions 1 & 2 — Building the Retrieval Foundation
**Submitted by:** Raju Boddula

---

## Scenario

We are on the **Data Dynamics** team (Anya, Ben, Clara). The company's internal knowledge —
onboarding guides, architecture notes, incident postmortems, model reports, security policy —
lives in scattered documents that nobody can search. Ask *"do I need approval to export customer
data?"* and you either know which document to open, or you ask in Slack and wait.

This notebook takes the first concrete step: an **AI-powered semantic search engine** over that
document collection, built from the components covered in Sessions 1 & 2.

## What this notebook delivers

| # | Assignment step | Where |
|---|-----------------|-------|
| 1 | Data loading & **chunking** (with strategy comparison + justification) | Part 1 |
| 2 | **Vectorization** with `all-MiniLM-L6-v2`, dimensionality noted | Part 2 |
| 3 | **Vector database** — Qdrant in-memory (raw client *and* LangChain wrapper) | Part 3 |
| 4 | **Semantic search** function (query → embedding → top-K chunks) | Part 4 |
| 5 | **Presenting results** for sample queries | Part 5 |
| ★ | *Extra:* contextual chunk headers + a **retrieval evaluation** that proves they help | Parts 1.5, 4.2 |
| ★ | *Bonus:* **BM25 keyword search** + head-to-head comparison + hybrid fusion | Part 6 |
| ★ | *Bonus:* **Command-line interface** | Part 7 |
| ★ | *Extra:* full **RAG loop** — grounded answers from the retrieved chunks | Part 8 |
| — | **Written deliverable** (all four required write-ups) | Part 9 |

## How to run

```bash
cd Week4
uv sync                       # installs everything listed in pyproject.toml
uv run jupyter lab            # then open homework.ipynb and Run All
```

Everything runs **locally and offline** except Part 8, which calls the Gemini API and needs
`GOOGLE_API_KEY` in `Week4/.env`. Part 8 degrades gracefully if the key is missing — retrieval
(the actual assignment) never depends on it.

## Part 0 — Setup

### Installation prerequisites

The pre-read's install line, kept commented because the `Week4` uv environment already provides
these. Uncomment if you are running in Colab or a bare virtualenv.

In [1]:
# !pip install -qq langchain-text-splitters langchain-community langchain-huggingface \
#     langchain-qdrant sentence-transformers qdrant-client numpy scikit-learn rank_bm25 tabulate

In [2]:
from __future__ import annotations

import os
import re
import textwrap
import time
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd

# --- Chunking (Session 1) --------------------------------------------------------------
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# --- Embeddings (Session 2) ------------------------------------------------------------
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

# --- Vector database (Session 2) --------------------------------------------------------
from qdrant_client import QdrantClient
from qdrant_client.http.models import CollectionStatus, Distance, PointStruct, VectorParams
from langchain_qdrant import QdrantVectorStore

# --- Keyword search (bonus) -------------------------------------------------------------
from rank_bm25 import BM25Okapi

pd.set_option('display.max_colwidth', 90)

# Keep HuggingFace/tokenizers quiet inside notebooks.
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

DOCS_DIR = Path('data_dynamics_docs')
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

print('Setup complete. Document folder exists:', DOCS_DIR.is_dir())

Setup complete. Document folder exists: True


---

## Part 1 — Data Loading & Preparation

### 1.1 Loading the internal document set

Five plain-text internal documents live in `Week4/data_dynamics_docs/`. They are deliberately
*different kinds* of document, because a knowledge-search tool that only works on one genre is not
much use:

| File | Kind | Why it is in the set |
|------|------|----------------------|
| `01_engineering_onboarding_guide.txt` | Process guide | Procedural "how do I…" questions |
| `02_data_platform_architecture.txt` | Technical note | Design/architecture questions |
| `03_incident_postmortem_INC-2026-041.txt` | Postmortem | Contains an exact ID + a timeline — the case where keyword search shines |
| `04_customer_churn_model_report.txt` | ML model report | Numbers, metrics, methodology |
| `05_data_security_and_governance_policy.txt` | Policy | Rules, approvals, retention — the highest-value questions |

Loading is plain Python file reading, as the assignment permits.

In [3]:
def load_documents(folder: Path) -> list[Document]:
    """Read every .txt file in `folder` into a LangChain Document.

    Metadata carries the source filename and a short human-readable title so that every
    retrieved chunk can be traced back to the document it came from — provenance is what
    makes a search result trustworthy.
    """
    documents = []
    for path in sorted(folder.glob('*.txt')):
        text = path.read_text(encoding='utf-8')
        documents.append(
            Document(
                page_content=text,
                metadata={
                    'source': path.name,
                    # First non-empty line of each file is its title.
                    'title': next(line.strip() for line in text.splitlines() if line.strip()),
                },
            )
        )
    return documents


documents = load_documents(DOCS_DIR)

inventory = pd.DataFrame([
    {
        'source': doc.metadata['source'],
        'characters': len(doc.page_content),
        'words': len(doc.page_content.split()),
        'title': doc.metadata['title'][:58],
    }
    for doc in documents
])

print(f'Loaded {len(documents)} documents, {inventory.characters.sum():,} characters total\n')
print(inventory.to_markdown(index=False))

Loaded 5 documents, 23,254 characters total

| source                                     |   characters |   words | title                                                      |
|:-------------------------------------------|-------------:|--------:|:-----------------------------------------------------------|
| 01_engineering_onboarding_guide.txt        |         4821 |     755 | DATA DYNAMICS - ENGINEERING ONBOARDING GUIDE               |
| 02_data_platform_architecture.txt          |         4507 |     714 | DATA DYNAMICS - INTERNAL TECHNICAL NOTE: PLATFORM ARCHITEC |
| 03_incident_postmortem_INC-2026-041.txt    |         4656 |     699 | DATA DYNAMICS - INCIDENT POSTMORTEM                        |
| 04_customer_churn_model_report.txt         |         4399 |     701 | DATA DYNAMICS - MODEL REPORT: CUSTOMER CHURN PREDICTION v2 |
| 05_data_security_and_governance_policy.txt |         4871 |     738 | DATA DYNAMICS - DATA SECURITY AND GOVERNANCE POLICY        |


In [4]:
# A peek at the raw text of one document, to see what the splitter is working with.
print(documents[2].page_content[:900], '...', sep='\n')

DATA DYNAMICS - INCIDENT POSTMORTEM
Incident ID: INC-2026-041
Title: Gold layer stale for 19 hours after silent Airbyte connector failure
Severity: SEV-2
Date of incident: 21 February 2026
Author: Ben Okafor
Reviewed by: Anya Raghavan, Clara Mendez
Status: Actions in progress

1. SUMMARY

Between 02:10 and 21:05 UTC on 21 February 2026, the gold layer tables for the Retail Analytics data
product served data that was up to 19 hours stale. Three client dashboards displayed yesterday's
numbers without any indication that the data was old. No data was lost or corrupted, and no
unauthorised access occurred. The cause was a failed Airbyte connector whose failure was not surfaced
because the connector's own retry loop kept the Airflow sensor in a waiting state rather than failing.

2. IMPACT

Two clients noticed and raised tickets before we did, which is the part of this incident we consider
mo
...


### 1.2 Why chunk at all?

A whole document is the wrong unit for retrieval, for two reasons covered in Session 1:

1. **Context-window limits.** Our documents are 4–6 KB each. Five of them would already crowd a
   prompt, and a real corpus is thousands of documents.
2. **Retrieval precision — the real reason.** An embedding is a *single* fixed-length vector. Ask
   one 5,000-character vector to represent onboarding *and* VPN setup *and* training courses *and*
   escalation paths, and it becomes an average of all of them — close to everything and specific to
   nothing. This is semantic dilution. Smaller chunks give sharper, more discriminative vectors.

The trade-off runs in both directions:

| | Chunks too small | Chunks too large |
|---|---|---|
| Vector quality | Sharp, focused | Diluted, "averaged" |
| Self-containedness | Answer gets cut in half | Answer is buried in noise |
| Cost | Many vectors to store/search | Few vectors, bloated prompts |

**Overlap** is the mitigation for the boundary problem: repeating the tail of chunk *n* at the head
of chunk *n+1* means a sentence straddling a cut still appears whole somewhere.

### 1.3 Comparing the two strategies

The assignment says *choose either* `CharacterTextSplitter` (fixed-size) or
`RecursiveCharacterTextSplitter`. Rather than assert a choice, let's measure both across a
parameter grid and pick on evidence.

In [5]:
def chunk_statistics(label: str, chunks: Sequence[Document]) -> dict:
    """Summarise a set of chunks so strategies can be compared numerically.

    `orphan_starts` counts chunks that begin mid-sentence (lowercase first character) — a cheap
    proxy for how often the splitter cut through the middle of a thought.
    """
    lengths = np.array([len(chunk.page_content) for chunk in chunks])
    orphans = sum(1 for chunk in chunks if chunk.page_content.lstrip()[:1].islower())
    return {
        'strategy': label,
        'chunks': len(chunks),
        'mean_len': int(lengths.mean()),
        'min_len': int(lengths.min()),
        'max_len': int(lengths.max()),
        'std_len': int(lengths.std()),
        'orphan_starts': orphans,
        'orphan_%': round(100 * orphans / len(chunks), 1),
    }


rows = []
for size, overlap in [(500, 50), (800, 120), (1200, 200)]:
    fixed = CharacterTextSplitter(
        separator='',           # a true fixed-size cut, ignoring text structure
        chunk_size=size,
        chunk_overlap=overlap,
        length_function=len,
    )
    recursive = RecursiveCharacterTextSplitter(
        separators=['\n\n', '\n', '. ', ' ', ''],   # paragraph -> line -> sentence -> word
        chunk_size=size,
        chunk_overlap=overlap,
        length_function=len,
    )
    rows.append(chunk_statistics(f'Fixed-size      ({size}/{overlap})', fixed.split_documents(documents)))
    rows.append(chunk_statistics(f'Recursive       ({size}/{overlap})', recursive.split_documents(documents)))

comparison = pd.DataFrame(rows).sort_values(['strategy']).reset_index(drop=True)
print(comparison.to_markdown(index=False))

| strategy                   |   chunks |   mean_len |   min_len |   max_len |   std_len |   orphan_starts |   orphan_% |
|:---------------------------|---------:|-----------:|----------:|----------:|----------:|----------------:|-----------:|
| Fixed-size      (1200/200) |       25 |       1089 |       398 |      1200 |       234 |              17 |       68   |
| Fixed-size      (500/50)   |       53 |        483 |       155 |       500 |        58 |              43 |       81.1 |
| Fixed-size      (800/120)  |       35 |        766 |       318 |       800 |       105 |              26 |       74.3 |
| Recursive       (1200/200) |       28 |        848 |       133 |      1183 |       299 |               0 |        0   |
| Recursive       (500/50)   |       70 |        335 |        19 |       494 |       127 |               4 |        5.7 |
| Recursive       (800/120)  |       40 |        592 |       258 |       798 |       141 |               1 |        2.5 |


In [6]:
# The difference is easiest to *see* at a chunk boundary. Same parameters, both splitters.
fixed_demo = CharacterTextSplitter(
    separator='', chunk_size=800, chunk_overlap=120, length_function=len
).split_documents(documents)
recursive_demo = RecursiveCharacterTextSplitter(
    separators=['\n\n', '\n', '. ', ' ', ''], chunk_size=800, chunk_overlap=120, length_function=len
).split_documents(documents)

for label, chunks in [('FIXED-SIZE', fixed_demo), ('RECURSIVE', recursive_demo)]:
    print(f'===== {label}: how chunk #3 starts and ends ' + '=' * 24)
    print('START >>>', repr(chunks[3].page_content[:110]))
    print('END   >>>', repr(chunks[3].page_content[-110:]))
    print()

===== FIXED-SIZE: how chunk #3 starts and ends ========================
START >>> 'ur\ndevice is authorised automatically once okta-grp-vpn-standard is approved. If a service times out,\ncheck th'
END   >>> 'n tests. Allocate at least 8 GB of memory to Docker or the test\nsuite will be killed by the out-of-memory reap'

===== RECURSIVE: how chunk #3 starts and ends ========================
START >>> 'VPN is required for anything that is not Slack, Workday or GitHub. The VPN client is Tailscale; your\ndevice is'
END   >>> ' This keeps the lockfile authoritative and avoids the drift that used to cause\n"works on my machine" failures.'



Fixed-size chunking slices at an arbitrary character offset, so chunks routinely open and close
mid-word. Recursive splitting tries `\n\n` first, falls back to `\n`, then `. `, then a space —
so it cuts at a paragraph break whenever one is available near the target size, and the chunk
stays a readable unit. The price is chunk sizes that vary rather than being uniform, which is a
cost we happily pay: **we index meaning, not fixed-width records.**

### 1.4 The chunking configuration we ship

**`RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)`** — full justification in
Part 9.

In [7]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120     # 15% of chunk_size

text_splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '\n', '. ', ' ', ''],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)

chunks = text_splitter.split_documents(documents)

# Number the chunks within each source document so results can cite "chunk 4 of 11".
per_source_counter: dict[str, int] = {}
for chunk in chunks:
    source = chunk.metadata['source']
    per_source_counter[source] = per_source_counter.get(source, 0) + 1
    chunk.metadata['chunk_index'] = per_source_counter[source]
for chunk in chunks:
    chunk.metadata['chunks_in_source'] = per_source_counter[chunk.metadata['source']]

print(f'{len(documents)} documents  ->  {len(chunks)} chunks')
print(f'Average chunk length: {np.mean([len(c.page_content) for c in chunks]):.0f} characters\n')
print(
    pd.Series([c.metadata['source'] for c in chunks])
    .value_counts()
    .sort_index()
    .rename('chunks')
    .to_markdown()
)

5 documents  ->  40 chunks
Average chunk length: 592 characters

|                                            |   chunks |
|:-------------------------------------------|---------:|
| 01_engineering_onboarding_guide.txt        |        8 |
| 02_data_platform_architecture.txt          |        8 |
| 03_incident_postmortem_INC-2026-041.txt    |        9 |
| 04_customer_churn_model_report.txt         |        7 |
| 05_data_security_and_governance_policy.txt |        8 |


In [8]:
# Inspect the first few chunks to verify the splitting works as expected.
for chunk in chunks[:3]:
    meta = chunk.metadata
    print('=' * 100)
    print(f"[{meta['source']}]  chunk {meta['chunk_index']}/{meta['chunks_in_source']}  "
          f"({len(chunk.page_content)} chars)")
    print('-' * 100)
    print(chunk.page_content)
    print()

[01_engineering_onboarding_guide.txt]  chunk 1/8  (552 chars)
----------------------------------------------------------------------------------------------------
DATA DYNAMICS - ENGINEERING ONBOARDING GUIDE
Document ID: DD-ENG-001
Owner: Anya Raghavan, Head of Data Engineering
Last revised: 12 January 2026
Audience: All new engineering hires (Data Engineer, Analytics Engineer, ML Engineer)

1. YOUR FIRST DAY

Welcome to Data Dynamics. Your first day is deliberately light on delivery work. IT will hand you a
company laptop that has already been enrolled in mobile device management. Do not install a personal
machine for client work; our client contracts forbid processing customer data on unmanaged hardware.

[01_engineering_onboarding_guide.txt]  chunk 2/8  (586 chars)
----------------------------------------------------------------------------------------------------
Before lunch you should have completed three things: signed the confidentiality acknowledgement in
Workday, enabled mult

### 1.5 One refinement: contextual chunk headers

Splitting creates a problem it does not solve. Chunk 1 of a document opens with the title, so it
knows what document it belongs to. **Chunk 6 does not.** Once split, a chunk reading *"Three
courses must be completed within your first 30 days…"* carries no trace of the fact that it comes
from the onboarding guide. The embedding therefore encodes "training courses" but not "onboarding",
and a query phrased around onboarding has nothing to match on.

The fix is cheap: **prefix every chunk with its document title before embedding it.** Each chunk
then carries both its local content and its global context. This is sometimes called a contextual
chunk header, and it is one of the highest value-per-line improvements available in a RAG pipeline.

We apply it to the text that is *actually indexed*, so what gets embedded and what gets displayed
stay identical — no hidden divergence between the index and what a user sees. Part 4.2 measures
whether it was worth doing rather than taking it on faith.

In [9]:
# Keep an un-prefixed copy so Part 4.2 can measure whether the headers actually helped.
chunks_without_headers = [
    Document(page_content=chunk.page_content, metadata=dict(chunk.metadata))
    for chunk in chunks
]

before = chunks[5].page_content

for chunk in chunks:
    title = chunk.metadata['title']
    # Chunk 1 of each document already opens with the title - don't repeat it.
    if not chunk.page_content.lstrip().startswith(title):
        chunk.page_content = f'{title}\n\n{chunk.page_content}'

print('BEFORE (no idea which document this chunk belongs to)')
print('-' * 100)
print(before[:230], '...\n')
print('AFTER (context restored)')
print('-' * 100)
print(chunks[5].page_content[:230], '...')

BEFORE (no idea which document this chunk belongs to)
----------------------------------------------------------------------------------------------------
4. HOW WE WORK

Sprints are two weeks and start on a Wednesday. Standup is asynchronous and posted in the team channel
by 10:00 local time. We keep a written record of decisions: anything that changes an interface, a
schema or a s ...

AFTER (context restored)
----------------------------------------------------------------------------------------------------
DATA DYNAMICS - ENGINEERING ONBOARDING GUIDE

4. HOW WE WORK

Sprints are two weeks and start on a Wednesday. Standup is asynchronous and posted in the team channel
by 10:00 local time. We keep a written record of decisions: anyth ...


---

## Part 2 — Vectorization: Creating Embeddings

### 2.1 The model

We use **`sentence-transformers/all-MiniLM-L6-v2`**, the model recommended in the pre-read:

- **384 dimensions** — small enough that similarity search over our corpus is instant, large enough
  to carry real semantic nuance. (Session 2 showed the 64-dim model returning noticeably weaker
  neighbours and the 768-dim model costing more for little gain at this scale.)
- **~80 MB, 6 transformer layers** — runs on CPU in a classroom laptop, no GPU and no API key.
- **Trained on 1B+ sentence pairs** for exactly this task: sentence-level semantic similarity.
- **Free and local** — no per-token cost and no client data leaving the machine, which matters
  given that document 05 is our own data-governance policy.

The critical rule: **the same model must embed both the documents and the query.** Two models
produce two incompatible vector spaces, and cosine similarity between them is meaningless.

In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={'normalize_embeddings': True},   # unit vectors -> cosine == dot product
)

started = time.perf_counter()
chunk_vectors = np.array(embedding_model.embed_documents([c.page_content for c in chunks]))
elapsed = time.perf_counter() - started

EMBEDDING_DIM = chunk_vectors.shape[1]

print(f'Embedded {len(chunks)} chunks in {elapsed:.2f}s '
      f'({1000 * elapsed / len(chunks):.0f} ms per chunk on CPU)')
print(f'Embedding matrix shape : {chunk_vectors.shape}')
print(f'DIMENSIONALITY         : {EMBEDDING_DIM}   <- required for the Qdrant collection')
print(f'Vector norm (sanity)   : {np.linalg.norm(chunk_vectors[0]):.4f}  (1.0 => normalised)')
print(f'\nFirst 8 values of chunk 0: {chunk_vectors[0][:8].round(4)}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedded 40 chunks in 0.20s (5 ms per chunk on CPU)
Embedding matrix shape : (40, 384)
DIMENSIONALITY         : 384   <- required for the Qdrant collection
Vector norm (sanity)   : 1.0000  (1.0 => normalised)

First 8 values of chunk 0: [-0.0488 -0.0589  0.0391  0.0028 -0.0228 -0.1229  0.0566 -0.0229]


### 2.2 Sanity check — does the vector space actually encode *meaning*?

Before trusting these vectors for retrieval, verify the property everything else rests on: texts
that mean similar things should land close together, **even when they share no vocabulary**.

In [11]:
probe_sentences = [
    'How do I get permission to write to the production database?',   # A
    'Production write access needs two approvals and a training course.',  # B - same meaning, different words
    'Requesting elevated privileges on the live system.',             # C - same meaning again
    'The churn model achieved an AUC of 0.847 on held-out data.',     # D - unrelated topic
]

probe_vectors = np.array(embedding_model.embed_documents(probe_sentences))
similarity_matrix = cosine_similarity(probe_vectors, probe_vectors)

labels = ['A: access question', 'B: access answer', 'C: access paraphrase', 'D: model metrics']
print(
    pd.DataFrame(similarity_matrix.round(3), index=labels, columns=['A', 'B', 'C', 'D'])
    .to_markdown()
)
print('\nA vs B (same meaning, almost no shared words):', round(similarity_matrix[0, 1], 3))
print('A vs D (different topic entirely)            :', round(similarity_matrix[0, 3], 3))

|                      |     A |     B |      C |      D |
|:---------------------|------:|------:|-------:|-------:|
| A: access question   | 1     | 0.444 |  0.272 |  0.026 |
| B: access answer     | 0.444 | 1     |  0.21  |  0.11  |
| C: access paraphrase | 0.272 | 0.21  |  1     | -0.035 |
| D: model metrics     | 0.026 | 0.11  | -0.035 |  1     |

A vs B (same meaning, almost no shared words): 0.444
A vs D (different topic entirely)            : 0.026


A and B score high while sharing barely a content word between them; A and D score low. That gap is
the entire value proposition of semantic search over keyword matching — and note that a BM25 index
would rank A against B near zero, because they overlap on almost nothing lexically.

---

## Part 3 — Storing Embeddings: The Vector Database

We use **Qdrant in in-memory mode** (`location=":memory:"`). Justification is in Part 9; the short
version is that Qdrant gives us real payloads, metadata filtering and a server-identical API for
zero setup cost.

Two ways of doing this are shown, because both appear in the sessions:

- **3.1 — the raw `qdrant-client`**, so the Collection / Point / `upsert` / `query_points`
  mechanics are explicit rather than hidden behind a wrapper.
- **3.2 — the `QdrantVectorStore` LangChain wrapper**, which is what the rest of the notebook uses.

### 3.1 The raw client — collections, points, upsert

The **critical requirement** called out in the pre-read: when creating an in-memory collection you
must state the **vector dimension explicitly**. Qdrant cannot infer it, and a mismatch between the
declared size and the vectors you upsert is a hard error at insert time.

In [12]:
COLLECTION_NAME = 'data_dynamics_docs'

client = QdrantClient(location=':memory:')      # no server, no Docker — runs in this process

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,          # 384 — MUST match the embedding model exactly
        distance=Distance.COSINE,    # cosine: compares direction (meaning), ignores magnitude
    ),
)

collection = client.get_collection(COLLECTION_NAME)
print(f'Collection "{COLLECTION_NAME}" created')
print(f'  status     : {collection.status}  (GREEN => ready)')
print(f'  dimension  : {EMBEDDING_DIM}')
print(f'  distance   : {Distance.COSINE}')
print(f'  is green   : {collection.status == CollectionStatus.GREEN}')

Collection "data_dynamics_docs" created
  status     : green  (GREEN => ready)
  dimension  : 384
  distance   : Cosine
  is green   : True


In [13]:
# A Point = id + vector + payload. The payload is what makes results human-readable:
# it carries the chunk text and its provenance alongside the raw numbers.
points = [
    PointStruct(
        id=index,
        vector=vector.tolist(),
        payload={
            'text': chunk.page_content,
            'source': chunk.metadata['source'],
            'title': chunk.metadata['title'],
            'chunk_index': chunk.metadata['chunk_index'],
        },
    )
    for index, (chunk, vector) in enumerate(zip(chunks, chunk_vectors))
]

client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)

print(f'Upserted {client.count(COLLECTION_NAME).count} points into "{COLLECTION_NAME}"')

Upserted 40 points into "data_dynamics_docs"


In [14]:
# A raw similarity search: embed the query with the SAME model, then query the collection.
raw_query = 'What happens if a client credential leaks into a git repository?'
raw_query_vector = embedding_model.embed_query(raw_query)

hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=raw_query_vector,
    limit=3,
    with_payload=True,
).points

print(f'Query: {raw_query}\n')
for rank, hit in enumerate(hits, start=1):
    print(f"{rank}. score={hit.score:.4f}  [{hit.payload['source']}]")
    print(textwrap.indent(textwrap.fill(hit.payload['text'][:260] + ' ...', 96), '     '))
    print()

Query: What happens if a client credential leaks into a git repository?

1. score=0.5790  [05_data_security_and_governance_policy.txt]
     DATA DYNAMICS - DATA SECURITY AND GOVERNANCE POLICY  5. SECRETS AND CREDENTIALS  Credentials
     live in the secrets manager and nowhere else. A credential committed to a repository is a
     reportable security incident even if the repository is private and even if th ...

2. score=0.4496  [01_engineering_onboarding_guide.txt]
     DATA DYNAMICS - ENGINEERING ONBOARDING GUIDE  Docker Desktop is licensed for all engineers and
     is required to run the local Postgres and MinIO containers that back the integration tests.
     Allocate at least 8 GB of memory to Docker or the test suite will be kill ...

3. score=0.3026  [03_incident_postmortem_INC-2026-041.txt]
     DATA DYNAMICS - INCIDENT POSTMORTEM  4. ROOT CAUSE  The direct trigger was an unannounced
     credential rotation by an upstream partner. The reason a routine credential problem becam

Notice the top hit surfaces the "committed to a repository is a reportable security incident"
passage from the policy document — retrieved by meaning. The query says "leaks into a git
repository"; the document says "committed to a repository … must be assumed compromised". Little
lexical overlap, correct answer.

### 3.2 The LangChain wrapper

`QdrantVectorStore` wraps exactly the same machinery — it creates the collection, embeds the
documents and upserts them — and hands back LangChain `Document` objects with their metadata
intact. This is what Parts 4–8 use.

In [15]:
vector_store = QdrantVectorStore.from_documents(
    chunks,
    embedding_model,
    collection_name='data_dynamics_langchain',
    location=':memory:',
)

print('QdrantVectorStore ready.')
print('Smoke test — "how long is data kept after a contract ends?"\n')
for doc in vector_store.similarity_search('how long is data kept after a contract ends?', k=2):
    print(f"  [{doc.metadata['source']}] {doc.page_content[:150].strip()} ...")

QdrantVectorStore ready.
Smoke test — "how long is data kept after a contract ends?"

  [05_data_security_and_governance_policy.txt] DATA DYNAMICS - DATA SECURITY AND GOVERNANCE POLICY

6. RETENTION AND DELETION

Retention periods are set per client in the engagement contract. Where ...
  [03_incident_postmortem_INC-2026-041.txt] DATA DYNAMICS - INCIDENT POSTMORTEM

5. WHAT WENT WELL

The backfill itself was clean and took under an hour once the credentials were fixed, which vi ...


---

## Part 4 — Implementing Retrieval (Semantic Search)

The retrieval function. It takes a user query, embeds it with **the same model** used for the
chunks, asks the vector store for the top-K nearest vectors, and returns the associated text
chunks with their similarity scores and provenance.

In [16]:
def semantic_search(query: str, k: int = 3, store: QdrantVectorStore = None) -> list[dict]:
    """Retrieve the k most semantically similar chunks for `query`.

    The query is embedded by the *same* model that embedded the chunks (LangChain handles this
    internally, using the embedding object the store was built with) and compared by cosine
    similarity. Scores are in [0, 1]; higher is more similar.
    """
    store = store or vector_store
    hits = store.similarity_search_with_score(query, k=k)
    return [
        {
            'rank': rank,
            'score': round(float(score), 4),
            'source': doc.metadata['source'],
            'chunk': f"{doc.metadata['chunk_index']}/{doc.metadata['chunks_in_source']}",
            'text': doc.page_content,
        }
        for rank, (doc, score) in enumerate(hits, start=1)
    ]


# Quick check that the contract holds.
sample = semantic_search('who approves access to personal data?', k=3)
print(pd.DataFrame(sample)[['rank', 'score', 'source', 'chunk']].to_markdown(index=False))

|   rank |   score | source                                     | chunk   |
|-------:|--------:|:-------------------------------------------|:--------|
|      1 |  0.6093 | 05_data_security_and_governance_policy.txt | 2/8     |
|      2 |  0.5889 | 05_data_security_and_governance_policy.txt | 5/8     |
|      3 |  0.4807 | 05_data_security_and_governance_policy.txt | 3/8     |


### 4.2 Does it actually work? A small retrieval evaluation

Eyeballing a couple of queries is not evidence. Below is a ten-query evaluation set — two questions
per source document, each phrased the way an employee would ask rather than the way the document is
written — scored on **top-1 document accuracy**: did the single best-scoring chunk come from the
document that genuinely holds the answer?

The same harness answers the open question from Part 1.5: were the contextual chunk headers worth
it? We build a second, header-free index and run the identical evaluation against it. This is an
**ablation** — change exactly one thing, measure the difference.

In [17]:
EVALUATION_SET = [
    ('What laptop and tools do I need on my first day?',        '01_engineering_onboarding_guide.txt'),
    ('What training do I have to complete?',                    '01_engineering_onboarding_guide.txt'),
    ('Where should deduplication logic live?',                  '02_data_platform_architecture.txt'),
    ('How is the platform kept from overspending?',             '02_data_platform_architecture.txt'),
    ('Why did the retail dashboards show stale numbers?',       '03_incident_postmortem_INC-2026-041.txt'),
    ('What caused the 19 hour outage?',                         '03_incident_postmortem_INC-2026-041.txt'),
    ('How accurate is the churn prediction model?',             '04_customer_churn_model_report.txt'),
    ('When does the model get retrained?',                      '04_customer_churn_model_report.txt'),
    ('How long do we keep client data?',                        '05_data_security_and_governance_policy.txt'),
    ('Who do I tell if I leaked something sensitive?',          '05_data_security_and_governance_policy.txt'),
]

# The ablation index: identical in every respect except the missing contextual headers.
baseline_store = QdrantVectorStore.from_documents(
    chunks_without_headers,
    embedding_model,
    collection_name='ablation_no_headers',
    location=':memory:',
)


def evaluate(store, label: str) -> pd.DataFrame:
    """Score a store on the evaluation set: top-1 and top-3 document accuracy."""
    rows = []
    for query, expected in EVALUATION_SET:
        results = semantic_search(query, k=3, store=store)
        retrieved = [result['source'] for result in results]
        rows.append({
            'query': query[:44],
            'expected': expected[:2],
            'got@1': retrieved[0][:2],
            'hit@1': retrieved[0] == expected,
            'hit@3': expected in retrieved,
            'score': results[0]['score'],
        })
    frame = pd.DataFrame(rows)
    frame.attrs['label'] = label
    return frame


with_headers = evaluate(vector_store, 'with contextual headers')
without_headers = evaluate(baseline_store, 'without contextual headers')

print('PER-QUERY RESULTS (index WITH contextual headers)\n')
print(with_headers.to_markdown(index=False))

PER-QUERY RESULTS (index WITH contextual headers)

| query                                        |   expected |   got@1 | hit@1   | hit@3   |   score |
|:---------------------------------------------|-----------:|--------:|:--------|:--------|--------:|
| What laptop and tools do I need on my first  |         01 |      01 | True    | True    |  0.236  |
| What training do I have to complete?         |         01 |      01 | True    | True    |  0.375  |
| Where should deduplication logic live?       |         02 |      02 | True    | True    |  0.4083 |
| How is the platform kept from overspending?  |         02 |      02 | True    | True    |  0.4005 |
| Why did the retail dashboards show stale num |         03 |      03 | True    | True    |  0.4785 |
| What caused the 19 hour outage?              |         03 |      03 | True    | True    |  0.4267 |
| How accurate is the churn prediction model?  |         04 |      04 | True    | True    |  0.6724 |
| When does the model get retra

In [18]:
summary = pd.DataFrame([
    {
        'index': label,
        'top-1 accuracy': f"{frame['hit@1'].sum()}/{len(frame)}",
        'top-3 accuracy': f"{frame['hit@3'].sum()}/{len(frame)}",
        'mean top-1 score': round(frame['score'].mean(), 4),
    }
    for label, frame in [('without contextual headers', without_headers),
                         ('with contextual headers', with_headers)]
])
print('ABLATION: does prefixing each chunk with its document title help?\n')
print(summary.to_markdown(index=False))

regressions = [
    query for (query, _), before, after
    in zip(EVALUATION_SET, without_headers['hit@1'], with_headers['hit@1'])
    if before and not after
]
improvements = [
    query for (query, _), before, after
    in zip(EVALUATION_SET, without_headers['hit@1'], with_headers['hit@1'])
    if after and not before
]
print('\nFixed by adding headers :', improvements or 'none')
print('Broken by adding headers:', regressions or 'none')

ABLATION: does prefixing each chunk with its document title help?

| index                      | top-1 accuracy   | top-3 accuracy   |   mean top-1 score |
|:---------------------------|:-----------------|:-----------------|-------------------:|
| without contextual headers | 9/10             | 10/10            |             0.4438 |
| with contextual headers    | 10/10            | 10/10            |             0.4354 |

Fixed by adding headers : ['What laptop and tools do I need on my first day?']
Broken by adding headers: none


Two things are worth taking from this. First, **the headers earn their place** — and the query they
fix is exactly the one predicted in Part 1.5, where the answer sat in a chunk that had been severed
from its document context. Second, and more important as a habit: this is a ten-line evaluation
harness that turns "the retrieval feels good" into a number you can regress against. Every future
change — a different chunk size, a bigger embedding model, hybrid retrieval — can now be *measured*
instead of argued about.

The honest caveat: ten queries over five documents is a smoke test, not a benchmark. Its value is
in catching outright breakage, not in ranking two systems that are close.

---

---

## Part 5 — Presenting Results

A small formatter, then the required demonstration: sample queries with the original query printed
alongside the text of the top-K retrieved chunks.

In [19]:
def show_results(query: str, results: list[dict], snippet_chars: int = 420, header: str = 'SEMANTIC SEARCH') -> None:
    """Print a query and its retrieved chunks in a readable, citable form."""
    print('=' * 100)
    print(f'{header}  |  QUERY: {query}')
    print('=' * 100)
    if not results:
        print('  (no results)\n')
        return
    for result in results:
        snippet = ' '.join(result['text'].split())
        if len(snippet) > snippet_chars:
            snippet = snippet[:snippet_chars].rsplit(' ', 1)[0] + ' ...'
        score = result.get('score')
        score_text = f"score {score:.4f}  |  " if isinstance(score, float) else ''
        print(f"\n  #{result['rank']}  {score_text}{result['source']}  (chunk {result['chunk']})")
        print(textwrap.indent(textwrap.fill(snippet, 92), '      '))
    print()


query = 'Can I use real production data in a test environment?'
show_results(query, semantic_search(query, k=3))

SEMANTIC SEARCH  |  QUERY: Can I use real production data in a test environment?

  #1  score 0.5069  |  05_data_security_and_governance_policy.txt  (chunk 4/8)
      DATA DYNAMICS - DATA SECURITY AND GOVERNANCE POLICY Personal data may not be used in
      development or test environments. Where realistic test data is needed, use the synthetic
      generation utility in the platform monorepo, which preserves distributions without
      reproducing real records. Masking production data is not an acceptable substitute, because
      masked data has repeatedly proved re-identifiable when joined against ...

  #2  score 0.3040  |  02_data_platform_architecture.txt  (chunk 6/8)
      DATA DYNAMICS - INTERNAL TECHNICAL NOTE: PLATFORM ARCHITECTURE 4. COST CONTROLS Snowflake
      warehouses are sized deliberately small and are configured to auto-suspend after 60 seconds
      of inactivity. The single largest source of unplanned spend in 2025 was a development
      warehouse left running b

In [20]:
# A spread of realistic questions an employee would actually ask.
sample_queries = [
    'Why did the retail dashboards show stale numbers?',            # -> postmortem
    'What laptop and tools do I need on my first day?',             # -> onboarding
    'How accurate is the churn prediction model?',                  # -> model report
    'Where should deduplication logic live?',                       # -> architecture note
]

for q in sample_queries:
    show_results(q, semantic_search(q, k=2), snippet_chars=300)

SEMANTIC SEARCH  |  QUERY: Why did the retail dashboards show stale numbers?

  #1  score 0.4785  |  03_incident_postmortem_INC-2026-041.txt  (chunk 1/9)
      DATA DYNAMICS - INCIDENT POSTMORTEM Incident ID: INC-2026-041 Title: Gold layer stale for 19
      hours after silent Airbyte connector failure Severity: SEV-2 Date of incident: 21 February
      2026 Author: Ben Okafor Reviewed by: Anya Raghavan, Clara Mendez Status: Actions in progress
      1. SUMMARY Between ...

  #2  score 0.4170  |  03_incident_postmortem_INC-2026-041.txt  (chunk 2/9)
      DATA DYNAMICS - INCIDENT POSTMORTEM 2. IMPACT Two clients noticed and raised tickets before
      we did, which is the part of this incident we consider most serious. The Retail Analytics
      REST API remained available throughout - availability was not affected - so our availability
      SLO was met while our ...

SEMANTIC SEARCH  |  QUERY: What laptop and tools do I need on my first day?

  #1  score 0.2360  |  01_engineering_onbo

The top hit for every one of these lands in the right document without sharing much vocabulary with
it — *"stale numbers"* retrieves the postmortem, *"where should deduplication logic live"* retrieves
the ingestion-vs-dbt rule of thumb, and the lead example above returns the exact clause forbidding
personal data in test environments. That is semantic retrieval doing its job; Part 4.2 confirms it
across the full evaluation set rather than on hand-picked examples.

---

## Part 6 — Bonus: Keyword Search (BM25) and a Head-to-Head Comparison

### 6.1 Implementing BM25

BM25 scores a document by how many query terms it contains, weighted by how rare each term is
across the corpus (IDF), with two refinements over plain TF-IDF: **term-frequency saturation**
(the 10th occurrence of a word adds far less than the 2nd) and **document-length normalisation**
(so long documents don't win by sheer size).

Tokenisation uses a regex plus scikit-learn's English stop-word list — no NLTK download required,
so the notebook stays runnable offline.

In [21]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

TOKEN_PATTERN = re.compile(r'[a-z0-9][a-z0-9\-]*')


def preprocess_text(text: str) -> list[str]:
    """Lowercase, tokenise to alphanumerics, drop English stop words.

    Hyphens are kept inside tokens so identifiers like 'inc-2026-041' and 'sev-2' survive intact —
    exactly the terms keyword search is best at.
    """
    tokens = TOKEN_PATTERN.findall(text.lower())
    return [token for token in tokens if token not in ENGLISH_STOP_WORDS]


bm25_corpus = [preprocess_text(chunk.page_content) for chunk in chunks]
bm25 = BM25Okapi(bm25_corpus)

print('Example tokenisation:')
print(' ', preprocess_text('Incident INC-2026-041 was a SEV-2 affecting the gold layer.'))
print(f'\nBM25 index built over {len(bm25_corpus)} chunks '
      f'({sum(len(t) for t in bm25_corpus):,} tokens, '
      f'{len(set().union(*bm25_corpus)):,} unique terms).')

Example tokenisation:
  ['incident', 'inc-2026-041', 'sev-2', 'affecting', 'gold', 'layer']

BM25 index built over 40 chunks (2,415 tokens, 1,065 unique terms).


In [22]:
def keyword_search(query: str, k: int = 3) -> list[dict]:
    """Retrieve the k highest-BM25-scoring chunks for `query`.

    Chunks scoring 0 share no meaningful term with the query and are dropped rather than padded
    into the result — an honest empty result beats an irrelevant one.
    """
    scores = bm25.get_scores(preprocess_text(query))
    top_indices = np.argsort(scores)[::-1][:k]
    return [
        {
            'rank': rank,
            'score': round(float(scores[index]), 4),
            'source': chunks[index].metadata['source'],
            'chunk': f"{chunks[index].metadata['chunk_index']}/{chunks[index].metadata['chunks_in_source']}",
            'text': chunks[index].page_content,
        }
        for rank, index in enumerate(top_indices, start=1)
        if scores[index] > 0
    ]


show_results('INC-2026-041', keyword_search('INC-2026-041', k=2), snippet_chars=300, header='BM25 KEYWORD SEARCH')

BM25 KEYWORD SEARCH  |  QUERY: INC-2026-041

  #1  score 2.6792  |  03_incident_postmortem_INC-2026-041.txt  (chunk 1/9)
      DATA DYNAMICS - INCIDENT POSTMORTEM Incident ID: INC-2026-041 Title: Gold layer stale for 19
      hours after silent Airbyte connector failure Severity: SEV-2 Date of incident: 21 February
      2026 Author: Ben Okafor Reviewed by: Anya Raghavan, Clara Mendez Status: Actions in progress
      1. SUMMARY Between ...



### 6.2 Head to head

Three query types, run through both retrievers. This is the comparison the deliverable asks for —
made empirically rather than asserted.

In [23]:
def compare_retrievers(query: str, k: int = 3) -> pd.DataFrame:
    """Show the top-k of both retrievers side by side for one query."""
    rows = []
    for label, results in [('semantic', semantic_search(query, k=k)), ('bm25', keyword_search(query, k=k))]:
        for result in results:
            rows.append({
                'retriever': label,
                'rank': result['rank'],
                'score': result['score'],
                'source': result['source'],
                'snippet': ' '.join(result['text'].split())[:70] + ' ...',
            })
        if not results:
            rows.append({'retriever': label, 'rank': '-', 'score': '-',
                         'source': '(no match)', 'snippet': 'BM25 found no overlapping terms'})
    return pd.DataFrame(rows)


test_queries = {
    'EXACT IDENTIFIER  (keyword should win)': 'INC-2026-041 SEV-2',
    'PARAPHRASE, NO SHARED WORDS  (semantic should win)':
        'I think I might have accidentally leaked something sensitive — who do I tell?',
    'MIXED: a rare term inside a natural question': 'what is the PSI threshold for retraining?',
}

for description, q in test_queries.items():
    print('#' * 100)
    print(f'# {description}')
    print(f'# QUERY: {q}')
    print('#' * 100)
    print(compare_retrievers(q, k=3).to_markdown(index=False))
    print()

####################################################################################################
# EXACT IDENTIFIER  (keyword should win)
# QUERY: INC-2026-041 SEV-2
####################################################################################################
| retriever   |   rank |   score | source                                  | snippet                                                                    |
|:------------|-------:|--------:|:----------------------------------------|:---------------------------------------------------------------------------|
| semantic    |      1 |  0.3362 | 03_incident_postmortem_INC-2026-041.txt | DATA DYNAMICS - INCIDENT POSTMORTEM 09:30 On-call engineer performs th ... |
| semantic    |      2 |  0.2895 | 03_incident_postmortem_INC-2026-041.txt | DATA DYNAMICS - INCIDENT POSTMORTEM ACTION-1 Cap Airbyte connector ret ... |
| semantic    |      3 |  0.2866 | 01_engineering_onboarding_guide.txt     | DATA DYNAMICS - ENGINEERING ONBOARDI

In [24]:
# The clearest single illustration: an intent expressed in words the document never uses.
paraphrase = 'I think I might have accidentally leaked something sensitive — who do I tell?'
show_results(paraphrase, semantic_search(paraphrase, k=1), snippet_chars=420, header='SEMANTIC')
show_results(paraphrase, keyword_search(paraphrase, k=1), snippet_chars=420, header='BM25')

SEMANTIC  |  QUERY: I think I might have accidentally leaked something sensitive — who do I tell?

  #1  score 0.3062  |  05_data_security_and_governance_policy.txt  (chunk 8/8)
      DATA DYNAMICS - DATA SECURITY AND GOVERNANCE POLICY 7. INCIDENT REPORTING Any suspected
      exposure of client data must be reported to the Data Protection Officer within one hour of
      discovery, through the #dd-security-incident channel or directly by phone out of hours. Do
      not attempt to assess severity before reporting - report first, assess second. Under GDPR we
      may have as little as 72 hours to notify a supervisory ...

BM25  |  QUERY: I think I might have accidentally leaked something sensitive — who do I tell?

  #1  score 4.8368  |  05_data_security_and_governance_policy.txt  (chunk 2/8)
      DATA DYNAMICS - DATA SECURITY AND GOVERNANCE POLICY PUBLIC Information already published by
      the client. No handling restrictions. INTERNAL Operational data with no personal or
      c

The semantic retriever finds the incident-reporting rule — *"report to the Data Protection Officer
within one hour of discovery"* — despite the query containing none of those words. BM25 latches
onto whichever chunk happens to repeat "sensitive", because lexical overlap is all it can see.

### 6.3 Hybrid retrieval — Reciprocal Rank Fusion

Neither retriever dominates, so in production you run both and fuse the rankings. **RRF** scores a
document by `Σ 1/(k + rank)` across result lists — it needs no score calibration between the two
systems, which is what makes it robust when one returns cosine similarities in [0,1] and the other
returns unbounded BM25 scores.

In [25]:
def hybrid_search(query: str, k: int = 3, pool: int = 10, rrf_k: int = 60) -> list[dict]:
    """Fuse semantic and BM25 rankings with Reciprocal Rank Fusion.

    Only the *ranks* from each retriever are used, never the raw scores, so the two incomparable
    score scales never have to be normalised against each other.
    """
    fused: dict[tuple[str, str], dict] = {}
    for results in (semantic_search(query, k=pool), keyword_search(query, k=pool)):
        for result in results:
            key = (result['source'], result['chunk'])
            entry = fused.setdefault(key, {'rrf': 0.0, 'result': result})
            entry['rrf'] += 1.0 / (rrf_k + result['rank'])

    ordered = sorted(fused.values(), key=lambda entry: entry['rrf'], reverse=True)[:k]
    return [
        {**entry['result'], 'rank': rank, 'score': round(entry['rrf'], 5)}
        for rank, entry in enumerate(ordered, start=1)
    ]


mixed_query = 'what is the PSI threshold for retraining?'
show_results(mixed_query, hybrid_search(mixed_query, k=3), snippet_chars=260,
             header='HYBRID (RRF: semantic + BM25)')

HYBRID (RRF: semantic + BM25)  |  QUERY: what is the PSI threshold for retraining?

  #1  score 0.0328  |  04_customer_churn_model_report.txt  (chunk 6/7)
      DATA DYNAMICS - MODEL REPORT: CUSTOMER CHURN PREDICTION v2 6. MONITORING AND RETRAINING
      Prediction drift is monitored weekly using population stability index on the score
      distribution; a PSI above 0.2 opens a ticket. Feature drift is monitored on the top ten ...

  #2  score 0.0161  |  03_incident_postmortem_INC-2026-041.txt  (chunk 9/9)
      DATA DYNAMICS - INCIDENT POSTMORTEM 7. LESSONS An unlimited retry policy is not resilience;
      it is a way of hiding failure. Any wait in our system must have a timeout, and any freshness
      objective must have a corresponding automated assertion, otherwise the ...

  #3  score 0.0161  |  04_customer_churn_model_report.txt  (chunk 5/7)
      DATA DYNAMICS - MODEL REPORT: CUSTOMER CHURN PREDICTION v2 5. FAIRNESS AND GOVERNANCE The
      model scores business accounts, not

---

## Part 7 — Bonus: A Command-Line Interface

The search wrapped in a small REPL. It supports switching retrieval mode at runtime
(`:mode semantic|keyword|hybrid`) and changing K (`:k 5`), so the comparison above can be explored
interactively rather than only read.

In [26]:
RETRIEVERS = {
    'semantic': semantic_search,
    'keyword': keyword_search,
    'hybrid': lambda query, k: hybrid_search(query, k=k),
}

BANNER = """
+----------------------------------------------------------------+
|      DATA DYNAMICS  -  Internal Knowledge Search                |
+----------------------------------------------------------------+
  Commands:  :mode semantic|keyword|hybrid   switch retriever
             :k <n>                          number of results
             :help                           show this banner
             :quit                           exit
  Anything else is treated as a search query.
"""


def run_query(query: str, mode: str, k: int) -> None:
    """Execute one query in `mode` and print the results."""
    show_results(query, RETRIEVERS[mode](query, k), snippet_chars=320, header=f'{mode.upper()} | top {k}')


def search_cli(scripted: Iterable[str] | None = None, mode: str = 'semantic', k: int = 3) -> None:
    """Interactive search REPL.

    Pass `scripted` (an iterable of input lines) to replay a session non-interactively — which is
    how the demo below runs, so this notebook executes end to end without blocking on stdin.
    """
    print(BANNER)
    scripted_inputs = iter(scripted) if scripted is not None else None

    while True:
        if scripted_inputs is None:
            try:
                line = input(f'[{mode}|k={k}] search> ').strip()
            except (EOFError, KeyboardInterrupt):
                print('\nBye.')
                return
        else:
            line = next(scripted_inputs, ':quit').strip()
            print(f'[{mode}|k={k}] search> {line}')

        if not line:
            continue
        if line in (':quit', ':q', 'exit'):
            print('Bye.')
            return
        if line == ':help':
            print(BANNER)
            continue
        if line.startswith(':mode '):
            requested = line.split(maxsplit=1)[1].strip()
            if requested in RETRIEVERS:
                mode = requested
                print(f'  -> retrieval mode set to "{mode}"\n')
            else:
                print(f'  !! unknown mode "{requested}". Choose from: {", ".join(RETRIEVERS)}\n')
            continue
        if line.startswith(':k '):
            value = line.split(maxsplit=1)[1].strip()
            if value.isdigit() and int(value) > 0:
                k = int(value)
                print(f'  -> returning top {k} results\n')
            else:
                print('  !! :k expects a positive integer\n')
            continue

        run_query(line, mode, k)

In [27]:
# Scripted demo session — the same code path a human would drive, replayed for reproducibility.
search_cli(scripted=[
    'How do I get production access?',
    ':mode keyword',
    'INC-2026-041',
    ':mode hybrid',
    ':k 2',
    'can I use real customer data for testing?',
    ':quit',
])


+----------------------------------------------------------------+
|      DATA DYNAMICS  -  Internal Knowledge Search                |
+----------------------------------------------------------------+
  Commands:  :mode semantic|keyword|hybrid   switch retriever
             :k <n>                          number of results
             :help                           show this banner
             :quit                           exit
  Anything else is treated as a search query.

[semantic|k=3] search> How do I get production access?
SEMANTIC | top 3  |  QUERY: How do I get production access?

  #1  score 0.4005  |  01_engineering_onboarding_guide.txt  (chunk 3/8)
      DATA DYNAMICS - ENGINEERING ONBOARDING GUIDE The standard bundle for a new data engineer is:
      - okta-grp-warehouse-reader (read access to the Snowflake analytics warehouse) - okta-grp-
      airflow-dev (author and trigger DAGs in the development environment) - okta-grp-github-dd
      (source control, read-write

To drive it yourself, run `search_cli()` with no arguments in a fresh cell (or from a terminal
after extracting these functions into a `.py` file) and type your own queries.

---

## Part 8 — Extra: Closing the Loop into a Full RAG Pipeline

Parts 1–7 build the **R** in RAG. This part adds **A**ugmentation and **G**eneration to show the
complete workflow: retrieve → stuff the chunks into a prompt → let the LLM answer *only* from
that context.

The prompt does two things that matter for a knowledge-base assistant:
1. It forbids answering outside the supplied context, which is what suppresses hallucination.
2. It requires the answer to cite its source documents, so a reader can verify the claim.

Needs `GOOGLE_API_KEY` in `Week4/.env`. **If the key is absent this section skips cleanly** — the
graded retrieval work above does not depend on it.

In [28]:
import warnings

from dotenv import load_dotenv

load_dotenv()

GEMINI_READY = False
try:
    # Same SDK as the classroom Rag.ipynb, so the two notebooks stay directly comparable.
    # `google-genai` is its successor; its deprecation notice is silenced, not acted on, here.
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', FutureWarning)
        import google.generativeai as genai
    from google.api_core.exceptions import ResourceExhausted

    api_key = os.getenv('GOOGLE_API_KEY')
    if api_key:
        genai.configure(api_key=api_key)
        MODEL_CANDIDATES = ['gemini-3-flash-preview', 'gemini-2.5-flash', 'gemini-flash-latest']
        GEMINI_READY = True
        print('Gemini configured. Candidate models:', ', '.join(MODEL_CANDIDATES))
    else:
        print('GOOGLE_API_KEY not found in the environment — Part 8 will be skipped.')
except ImportError:
    print('google-generativeai is not installed — Part 8 will be skipped.')

Gemini configured. Candidate models: gemini-3-flash-preview, gemini-2.5-flash, gemini-flash-latest


In [29]:
def generate_content(prompt_text: str, max_retries: int = 2):
    """Call Gemini with retry-on-429 and fallback to the next candidate model.

    The free tier gives each model its own small quota, so a 429 on the preferred model is normal
    rather than exceptional; we honour the retry delay the API asks for, then move down the list.
    """
    candidates = list(MODEL_CANDIDATES)
    last_error = None

    while candidates:
        name = candidates[0]
        model = genai.GenerativeModel(name)
        for attempt in range(max_retries):
            try:
                return model.generate_content(prompt_text)
            except ResourceExhausted as error:
                last_error = error
                asked = re.search(r'retry in ([0-9.]+)s', str(error))
                wait = float(asked.group(1)) + 1 if asked else 8 * (attempt + 1)
                print(f'{name}: rate limited, retrying in {wait:.0f}s ({attempt + 1}/{max_retries})')
                time.sleep(wait)
        print(f'{name}: quota exhausted, falling back to the next model')
        candidates.pop(0)

    raise last_error


RAG_PROMPT = """You are the Data Dynamics internal knowledge assistant.

Answer the employee's question using ONLY the context below, which was retrieved from our internal
documents. Follow these rules strictly:
- If the context does not contain the answer, say "I don't know based on our internal documents"
  and suggest who to ask. Never rely on outside knowledge.
- Cite the source filename in square brackets after each claim, e.g. [05_data_security_and_governance_policy.txt].
- Be concise and practical. Use bullet points for steps or requirements.

QUESTION: {question}

CONTEXT:
{context}
"""


def build_context(results: list[dict]) -> str:
    """Format retrieved chunks into a labelled context block for the prompt."""
    return '\n\n'.join(
        f"[source: {result['source']} | chunk {result['chunk']}]\n{result['text']}"
        for result in results
    )


def answer_question(question: str, k: int = 4, retriever=hybrid_search, show_context: bool = False) -> str | None:
    """Full RAG: retrieve -> augment the prompt -> generate a grounded answer."""
    if not GEMINI_READY:
        print('Skipped: GOOGLE_API_KEY is not configured.')
        return None

    results = retriever(question, k=k)
    context = build_context(results)

    if show_context:
        print('--- RETRIEVED CONTEXT ---')
        print(context[:1200], '...\n')

    response = generate_content(RAG_PROMPT.format(question=question, context=context))

    print('=' * 100)
    print('QUESTION:', question)
    print(f"RETRIEVED FROM: {', '.join(sorted({r['source'] for r in results}))}")
    print('=' * 100)
    print(response.text)
    return response.text

In [30]:
_ = answer_question('I am new here. What do I need to do before I can write to production?', k=4)

QUESTION: I am new here. What do I need to do before I can write to production?
RETRIEVED FROM: 01_engineering_onboarding_guide.txt
To gain production write access as a new engineer, you must complete the following requirements:

*   **Ship two changes to development:** You must successfully deploy two changes in the development environment before applying for production access [01_engineering_onboarding_guide.txt].
*   **Complete training:** You must finish the "Data Handling and Client Confidentiality" course (90 minutes). It is recommended to schedule this in your first week, as production access requests will remain pending until completion is automatically recorded [01_engineering_onboarding_guide.txt].
*   **Obtain approvals:** Once prerequisites are met, your request for production write access must be approved by both the platform owner and the security team. This process typically takes five to seven working days [01_engineering_onboarding_guide.txt].
*   **Use managed hardwar

In [31]:
# Guardrail check: a question the corpus genuinely cannot answer. A well-grounded RAG system
# should decline rather than invent — this is the anti-hallucination property in action.
_ = answer_question('What is the company holiday policy in Japan?', k=3)

QUESTION: What is the company holiday policy in Japan?
RETRIEVED FROM: 04_customer_churn_model_report.txt, 05_data_security_and_governance_policy.txt
I don't know based on our internal documents. Please contact the Human Resources department or refer to the local employee handbook for Japan.


---

# Part 9 — Written Deliverable

## 9.1 Choice of chunking strategy and parameters

**Chosen: `RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)`.**

**Why recursive over fixed-size.** The measurement in Part 1.3 makes the case. Fixed-size splitting
cuts at an arbitrary character offset — the boundary demo shows chunks opening and closing
mid-word, and the `orphan_starts` column counts how often a chunk begins mid-sentence. Recursive
splitting walks a separator hierarchy (`\n\n` → `\n` → `. ` → `" "` → `""`), so it prefers a
paragraph break near the target size and only falls back to a harder cut when it must. Our
documents are strongly paragraph-structured — numbered policy sections, a timeline, a corrective
actions list — so there is almost always a natural boundary to snap to.

This matters more than tidiness. A chunk that ends mid-sentence produces a *diluted embedding*: the
vector represents a fragment of a thought plus the beginning of an unrelated one. Since the
embedding is the only thing retrieval can see, a semantically incoherent chunk is a permanently
unfindable chunk. Fixed-size chunking is defensible for uniform, unstructured text; for structured
internal documents it throws away free signal.

The cost is variable chunk lengths (higher `std_len` in the comparison table). That is the right
trade: we index *meaning*, not fixed-width records.

**Why `chunk_size=800`.** Roughly 120–160 words, which lands close to one policy clause or one
postmortem section. The tension:

- Smaller (≈300): sharper vectors, but answers get split across chunks. The production-access rule
  — two approvals, a training prerequisite, quarterly review — would be spread over two or three
  chunks, and retrieving one gives an incomplete answer.
- Larger (≈2000): each chunk holds several unrelated topics, so its embedding averages them and
  becomes non-specific. Session 2's chunked-vs-unchunked Indian-food comparison showed exactly this
  failure — the unchunked store retrieved plausible-looking but less targeted results.

800 keeps most sections intact while staying focused, and it sits comfortably inside
`all-MiniLM-L6-v2`'s 256-token window — an important practical point, since text beyond that limit
is silently truncated and would simply never be searchable.

**One addition beyond the brief: contextual chunk headers (Part 1.5).** Splitting destroys context
that splitting cannot restore — chunk 6 of the onboarding guide has no idea it is part of the
onboarding guide. Prefixing each chunk with its document title before embedding fixes this in one
line, and the ablation in Part 4.2 shows it converting a top-1 miss into a hit on exactly the query
that failure predicts. This is the kind of change that is easy to adopt on faith and hard to
justify without a measurement, which is why it is measured rather than asserted.

**Why `chunk_overlap=120` (15%).** Overlap insures against the boundary problem: a sentence
straddling a cut appears whole in at least one chunk. 10–20% is the standard guidance; 15% covers
about one long sentence of run-on context at 800 characters. Going higher inflates the index with
near-duplicate vectors, which both costs storage and lets the same passage occupy several slots in
a top-K result — crowding out genuinely different material.

## 9.2 Choice of vector store

**Chosen: Qdrant in in-memory mode (`location=":memory:"`).**

| | FAISS | Qdrant (in-memory) |
|---|---|---|
| What it is | ANN **library** — an index of vectors | Vector **database engine** |
| Payloads | External — you keep a parallel `id → text` map yourself | Native JSON payload stored with each point |
| Metadata filtering | Not supported; filter after retrieval | First-class pre-filtering (`source`, date, classification…) |
| Path to production | Rewrite around a database | Same client, change one argument to a URL |
| Setup cost here | Zero | Zero — no server, no Docker |

Three reasons drove the choice:

1. **Payloads keep provenance attached to the vector.** With FAISS I would maintain a side-car
   dictionary mapping row indices back to text and filenames, and any reindexing risks the two
   drifting apart. Qdrant stores `text`, `source` and `chunk_index` *with* the point, which is why
   every result in this notebook can cite the document it came from. For an internal knowledge tool
   that is not a nicety — an uncitable answer is an untrustworthy one.

2. **Metadata filtering is where this goes next.** The obvious follow-up features — restrict search
   to the security policy, or exclude RESTRICTED-classified documents from a given user's results —
   are native Qdrant filters applied *during* the ANN search. In FAISS they would be post-hoc
   filtering of the top-K, which silently returns fewer results than requested and can return none
   at all.

3. **Identical API from laptop to cluster.** `QdrantClient(location=':memory:')` becomes
   `QdrantClient(url='http://localhost:6333')` — one line. Everything else, including the
   `QdrantVectorStore` wrapper, is unchanged. FAISS at production scale means adopting a different
   system entirely.

FAISS remains an excellent choice when you need raw speed over millions of vectors inside a single
process and already have your own storage layer. At our scale (tens of thousands of chunks) both
answer in milliseconds, so the tiebreaker is the operational features — and there Qdrant wins.

## 9.3 Semantic search vs keyword search — what each does well

The comparison in Part 6.2 was run on three deliberately chosen query types. The results:

**Semantic search wins on intent expressed in the user's own words.** The query *"I think I might
have accidentally leaked something sensitive — who do I tell?"* retrieved the incident-reporting
clause (*"report to the Data Protection Officer within one hour of discovery"*) — a passage that
contains neither "leaked" nor "accidentally" nor "tell". BM25 cannot make that jump; lexical
overlap is the only signal it has. This generalises to the whole class of query that matters most
for internal search: a new employee asks in the vocabulary they *have*, not the vocabulary the
policy author used. Semantic retrieval also absorbs synonyms ("cancel" / "churn"), paraphrase, and
question-shaped queries where the answer is phrased as a statement.

**Keyword search wins on exact tokens.** The query `INC-2026-041 SEV-2` is the clearest case. Those
identifiers are rare, high-IDF strings, and BM25 scores them decisively. Embeddings are the wrong
tool here: an identifier has no semantic neighbourhood, so `INC-2026-041` and `INC-2026-042` sit
almost on top of each other in vector space despite referring to different incidents. The same
applies to product SKUs, error codes, table names, ticket numbers, function names and people's
names. BM25 is also fully explainable (you can point at *which* term scored) and needs no model,
no GPU and no embedding step — a real advantage when a corpus changes constantly.

**Where each fails.** Semantic search fails on rare literal tokens and gives no explanation for
*why* something ranked highly; it also always returns K results, even when nothing is relevant,
because "nearest" is not the same as "near". BM25 fails the moment the query and the document use
different words for the same thing — the vocabulary-mismatch problem — and returns nothing at all
for a well-phrased question with no lexical overlap, which is a common outcome above.

**Therefore: hybrid.** Part 6.3 fuses both with Reciprocal Rank Fusion, which combines *ranks*
rather than scores and so sidesteps the fact that cosine similarity in [0,1] and unbounded BM25
scores are not comparable. On the mixed query *"what is the PSI threshold for retraining?"* —
a natural question containing one rare technical token — hybrid retrieval gets the benefit of both:
BM25 anchors on "PSI", the embedding understands "threshold for retraining". This is why production
RAG systems almost always run hybrid retrieval rather than either alone.

## 9.4 Retrieved chunks for a sample query

Query: **"How do I report a suspected data breach?"** — printed in full below. Note that the
corpus never uses the word *"breach"*; the policy says *"suspected exposure"*. A keyword index
would score the query's most important term at zero.

The top chunk carries the direct answer — report to the Data Protection Officer within one hour of
discovery, report before assessing severity, and note the 72-hour GDPR clock that starts at
discovery. The query and the document share almost no vocabulary: *"breach"* → *"suspected
exposure"*, *"report"* → *"reported"*, *"how do I"* → an imperative policy clause. That gap is
precisely what the embedding closes.

---

## Summary

| Assignment requirement | Status |
|---|---|
| Load 3–5 text documents | ✅ 5 internal Data Dynamics documents |
| Implement chunking, experiment with parameters, justify | ✅ Both splitters measured across a 3-point grid; recursive @ 800/120 chosen and justified |
| Inspect the first few chunks | ✅ Part 1.4 |
| Embed with `all-MiniLM-L6-v2` | ✅ Part 2, via `HuggingFaceEmbeddings` |
| Note the dimensionality | ✅ **384** |
| Vector DB in memory + justify the choice | ✅ Qdrant `:memory:`, raw client *and* LangChain wrapper |
| Explicit vector dimension on the collection | ✅ `VectorParams(size=384, distance=COSINE)` |
| Store vectors with text + metadata | ✅ Qdrant payloads carry text, source, chunk index |
| Semantic search function, same embedding model, top-K | ✅ `semantic_search()` |
| Print query + top-K chunk text | ✅ Parts 5 and 9.4 |
| **Bonus:** BM25 keyword search + when it wins | ✅ Part 6, measured head-to-head |
| **Bonus:** CLI or web UI | ✅ Part 7, `search_cli()` with mode/K commands |
| Write-up: chunking, vector store, search comparison, sample results | ✅ Part 9 |

**Beyond the brief:** contextual chunk headers with an ablation proving they help (1.5 + 4.2), a
ten-query retrieval evaluation harness (4.2), hybrid retrieval with Reciprocal Rank Fusion (6.3),
and a complete retrieve → augment → generate RAG loop with source citations and a hallucination
guardrail (Part 8).